# Notebook 04 - Regularization

In the previous notebook we met overfitting. When a model is too complex, it stops learning the real pattern and instead memorizes the training data, noise and all. It then looks great on the training set but does poorly on the future data we actually care about.

We saw one cure there, which was to pick a simpler model. But that is a blunt instrument. Often we want a flexible model, because it may genuinely need that flexibility to capture the real pattern, while still stopping it from going wild and overfitting to the noise. In this notebook we'll see a sharper tool called **regularization**. The main goal of regularization is to discourage a model from becoming too complex from inside the training process itself.

## Feature engineering and polynomial models

Back in Notebook 02 our model was a straight line, $f(x) = w_0 + w_1 x$, built from a single feature $x$. A straight line is rigid, real-world tasks are rarely linear, so we often want something more flexible. Remember that we mentioned existence of nonlinear models (and in fact seen examples in previous notebooks). One way to get a nonlinear model is feature engineering, where we build extra input columns out of the data we already have and let the model learn a weight for each one.

The classic example is polynomial features. Suppose two features $x_1$ and $x_2$ are available to the model (e.g. `size` and `number_of_bedrooms`). We can build new features out of them. Here we show three:

$$ x_3 = x_1^2, \quad x_4 = x_1 x_2, \quad x_5 = x_2^2. $$

Then we can build a model out of all five features:
$$f(x_1, x_2, x_3, x_4, x_5) = w_0 + w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 + w_5 x_5$$

Such models are generally called **higher-degree polynomial models**. A linear model is just a *degree-1* polynomial. Higher-degree polynomials can bend into curves even though it is still just a weighted sum of features. Let us bring our house prices dataset once again to see polynomial features in action on some house data. We again have a single feature `size` which we denote by $x$. In addition, we have two more hand-crafted features: $x^2$ and $x^3$. The model is then:
$$ f(x) = w_0 + w_1 x + w_2 x^2 + w_3 x^3. $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

size  = np.sort(rng.uniform(600, 3000, 40))
price = 40 + 0.03 * size + 2e-5 * size**2 + rng.normal(0, 15, size.shape)

is_test = rng.random(size.shape) < 0.3
size_tr, price_tr = size[~is_test], price[~is_test]
size_te, price_te = size[is_test],  price[is_test]

price_te = price_te + rng.normal(0, 20, price_te.shape)

line = np.polyfit(size_tr, price_tr, 1)
poly = np.polyfit(size_tr, price_tr, 3)
grid = np.linspace(600, 3000, 300)

plt.figure(figsize=(7, 4.6))
plt.scatter(size_tr, price_tr, color="tab:blue", alpha=0.6, label="train data")
plt.scatter(size_te, price_te, color="tab:orange", alpha=0.8, label="test data")
plt.plot(grid, np.polyval(line, grid), color="tab:red",   lw=2, label="straight line $(x)$")
plt.plot(grid, np.polyval(poly, grid), color="tab:green", lw=2, label="polynomial $(x, x^2, x^3)$")
plt.xlabel("size (square feet)"); plt.ylabel("price ($1000s)"); plt.legend()
plt.title("Feature engineering lets a model bend into a curve")
plt.show()

## Why we need regularization: complexity breeds overfitting

Specifically, the degree of the polynomial controls how much it can bend. A degree-1 model is a straight line. A degree-3 model can have a couple of turns. A degree-15 model can wiggle violently. So the polynomial degree is a clean dial for model complexity: turn it up and the model gets more complex (and flexible).

The natural question is, should we just turn that dial all the way up? More flexibility can only help the model fit the data, right? We should already be skeptical because of what we know about overfitting, but let us test that idea honestly. We will take one fixed dataset, fit polynomials of increasing degree to it, and measure two things for each:

- the training MSE, the error on the data the model learned from, and
- the test MSE, the error on fresh data it has never seen, which is the number we actually care about.

> **Hint:** Once again, remember that we use test set to imitate the future (what we'll encounter when we deploy the model).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(1)
true_price = lambda s: 40 + 0.03 * s + 2e-5 * s**2

size_tr  = np.sort(rng.uniform(600, 3000, 18))
price_tr = true_price(size_tr) + rng.normal(0, 18, size_tr.shape)
size_te  = np.sort(rng.uniform(600, 3000, 400))
price_te = true_price(size_te) + rng.normal(0, 18, size_te.shape)

def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(), LinearRegression())

def mse(model, x, y):
    return np.mean((model.predict(x.reshape(-1, 1)) - y) ** 2)

degrees = [1, 3, 6, 9]
grid = np.linspace(600, 3000, 300)
train_mses, test_mses = [], []

fig, axes = plt.subplots(1, len(degrees), figsize=(18, 3.4), sharey=True)
print(f"{'degree':>6} {'train MSE':>11} {'test MSE':>14}")
for ax, d in zip(axes, degrees):
    m = poly_model(d).fit(size_tr.reshape(-1, 1), price_tr)
    tr, te = mse(m, size_tr, price_tr), mse(m, size_te, price_te)
    train_mses.append(tr); test_mses.append(te)
    print(f"{d:>6} {tr:>11.0f} {te:>14,.0f}")
    ax.scatter(size_te, price_te, color="tab:orange", alpha=0.25, s=8,  zorder=2, label="test data")
    ax.scatter(size_tr, price_tr, color="tab:blue",  alpha=0.85, s=22, zorder=3, label="train data")
    ax.plot(grid, m.predict(grid.reshape(-1, 1)), color="tab:red", lw=2, zorder=4, label="model fit")
    ax.set_title(f"degree {d}\ntest MSE {te:,.0f}")
    ax.set_xlabel("size")
axes[0].set_ylabel("price ($1000s)")
axes[0].set_ylim(0, 300)
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4.4))
plt.plot(degrees, train_mses, "o-", color="tab:blue", label="train MSE")
plt.plot(degrees, test_mses,  "o-", color="tab:orange",    label="test MSE")
plt.yscale("log"); plt.xlabel("polynomial degree (model complexity)")
plt.ylabel("MSE (log scale)"); plt.legend()
plt.title("Train error keeps falling, but test error explodes once the model overfits")
plt.show()

Observe the following two behaviors. The training MSE decreases as model complexity increases because more complex models can always fit the available data better. However, this does not guarantee good performance on the test set. In fact, we can clearly see that the test MSE initially decreases as simpler models capture the underlying pattern, but then begins to increase as the models become more complex and start fitting noise in the training data.

That said, the takeaway is not that we should never use complex models. The true relationship in the data may itself be complex, and a simple model may never capture it adequately. However, we need a mechanism to prevent the model from treating random fluctuations in the training data as meaningful patterns. Regularization is one tool for addressing this problem.


## The core idea of regularization: keep the weights small

Before we write down any formulas, let us pin down the single idea behind most of the regularization methods. We just saw a flexible model overfit by chasing noise, but how does it manage to do that? To whip up and down through every training point, the wild degree-9 curve needs steep slopes and sharp turns, and the only way a polynomial produces steep slopes and sharp turns is by using *very large weights*. A model with small weights simply cannot produce wild, jagged curves, whereas a model with large weights can. So if we discourage large weights, we exactly the wiggly, noise-chasing behavior that causes overfitting, and we get to do it without lowering the model's degree at all. So we keep the flexible model and just keep its weights small.

The plot below makes this concrete. We hold the model fixed at a degree-9 polynomial and give it the same noisy points twice: once letting the weights grow as large as they like, and once keeping them small. Watch what the size of the weights does to the shape of the curve.

In [ ]:
rng = np.random.default_rng(3)

x  = np.linspace(-1, 1, 12)
y  = 0.5 * x + 0.4 + rng.normal(0, 0.08, x.shape)
xg = np.linspace(-1, 1, 400)

def design(t):
    return np.stack([t**k for k in range(10)], axis=1)

X, Xg = design(x), design(xg)

w_large, *_ = np.linalg.lstsq(X, y, rcond=None)
w_small = np.linalg.solve(X.T @ X + 0.1 * np.eye(X.shape[1]), X.T @ y)

fig, (axc, axw) = plt.subplots(1, 2, figsize=(13, 4.6))

axc.scatter(x, y, color="black", zorder=3, label="noisy data")
axc.plot(xg, Xg @ w_large, color="tab:red",   lw=2, label="large weights (regularization: no)")
axc.plot(xg, Xg @ w_small, color="tab:brown", lw=2, label="small weights (regularization: yes)")
axc.set_ylim(y.min() - 0.5, y.max() + 0.5)
axc.set_xlabel("x"); axc.set_ylabel("model output  f(x)")
axc.set_title("Same degree-9 model, two weight settings")
axc.legend()

powers = np.arange(10)
axw.bar(powers - 0.2, np.abs(w_large), width=0.4, color="tab:red",   label="large weights")
axw.bar(powers + 0.2, np.abs(w_small), width=0.4, color="tab:brown", label="small weights")
axw.set_yscale("log")
axw.set_xlabel("polynomial term (power of x)"); axw.set_ylabel("|weight|  (log scale)")
axw.set_title("The wild curve is built from large weights")
axw.legend()
plt.tight_layout(); plt.show()

print(f"{'No regularization:':<20}{'Regularization:'}")
for k, w in enumerate(w_large):
    print(f"{f'w_{k}: {w:.3f}':<20}{f'w_{k}: {w_small[k]:.3f}'}")

### Why small weights mean a tamer model

The bar chart on the right tells most of the story. To push a degree-9 curve through every noisy point, the unpenalized fit had to make the high-degree weights huge, with alternating signs. Each term like $x^7$ or $x^9$ is tiny on its own for $x$ between $-1$ and $1$ (for example, at $x = 0.5$ we get $x^9 \approx 0.002$). So the only way for such a term to have any effect is to multiply it by a giant weight. These giant terms mostly cancel each other out near the data points, but in between they don't, and that leftover is the wiggle.

Penalizing the size of the weights stops this from happening, and there is a simple reason why. The prediction is just a weighted sum of the features, $f(x) = \sum_j w_j\, x^j$, so the difference between the curve's value at any two inputs $a$ and $b$ is
$$ |f(a) - f(b)| = \Big|\sum_j w_j\,(a^j - b^j)\Big| \;\le\; \Big(\max_j |w_j|\Big)\sum_j |a^j - b^j|. $$
The right-hand side gets smaller as the weights get smaller. So when every weight is small, the curve can't change very much between nearby points, and it doesn't have enough room to make large up-and-down movements. It is forced to stay close to a smoother shape. Large weights take away that limit, and only then can the curve shoot up and drop down to chase the noise.

That is the whole idea. Instead of trying to guess the right degree, we let the model keep all of its flexibility but add a term to the loss that **penalizes large weights**:
$$ \mathcal{L} = \underbrace{\frac{1}{N}\sum_i (y_i - \hat{y}_i)^2}_{\text{fit the data}} \;+\; \lambda \underbrace{\,\text{penalty}(\mathbf{w})}_{\text{keep weights small}}. $$

Training is now a tug-of-war. The first term still wants to fit the data, even if that means big wiggly weights, while the second term pushes every weight toward zero. The knob $\lambda$ (the greek letter "lambda", often called the **weight decay coefficient**) decides which side wins. What "$\mathrm{penalty}$" does to the weights is the only difference between the regularization methods we will explore below.

We can revise our update rule as:
$$
\begin{aligned}
\mathbf{w}
&\leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}} \mathcal{L} \\
&= \mathbf{w} - \eta \nabla_{\mathbf{w}} \Bigg(
\frac{1}{N}\sum_i (y_i - \hat{y}_i)^2
+ \lambda\,\text{penalty}(\mathbf{w})
\Bigg) \\
&= \mathbf{w}
- \eta \Bigg[
\nabla_{\mathbf{w}} \left(\frac{1}{N}\sum_i (y_i - \hat{y}_i)^2\right)
+ \lambda \nabla_{\mathbf{w}}\,\text{penalty}(\mathbf{w})
\Bigg].
\end{aligned}
$$

> **Note 1:** Now we have another *hyperparameter* to tune! Often, learning rate ($\eta$) and weight decay coefficient ($\lambda$) are the two most important hyperparameters to tune in a model. 

> **Note 2:** We have shown the update rule for the **coupled** version of regularization, where the penalty is part of the loss and the update is a single step. There is also a **decoupled** version, where we first take a step to fit the data and then take a separate step to shrink the weights. The idea in both is the same, though the latter is more commonly used.

### L2 regularization, also called Ridge

L2 regularization uses the sum of squared weights as its penalty:

$$
\mathrm{penalty}(\mathbf{w}) = \sum_j w_j^{\,2} \\
\mathcal{L} \;=\; \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2 \;+\; \lambda \sum_{j} w_j^{\,2} 
$$

Squaring means a weight of $10$ costs $100$ while a weight of $1$ costs only $1$, so L2 comes down hardest on the largest weights and squashes them toward zero first. It shrinks every weight smoothly, but as we will see it almost never sets a weight exactly to zero. L2 is also known as **Ridge regression**.

### L1 regularization, also called Lasso

L1 regularization instead uses the sum of absolute values of the weights,

$$
\mathrm{penalty}(\mathbf{w}) = \sum_j |w_j| \\ 
\mathcal{L} \;=\; \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2 \;+\; \lambda \sum_{j} |w_j| 
$$

This small change has a striking effect, in that L1 drives the weights of unhelpful features all the way to exactly zero. A zero weight removes its feature completely, so L1 does not merely shrink a useless feature, it deletes it. The property is called **sparsity**, and it makes L1, also known as **Lasso regression**, a tool for **feature selection**, because it keeps the features that matter and zeroes out the rest.

>**Note:** Real-world datasets are rarely perfect. *Feature selection* helps identify which features are actually useful for making predictions. For example, a house price dataset might contain hundreds of features, including irrelevant ones such as `max_ticket_price_in_nba_finals` or `number_of_kittens_owned_by_professor`. Sometimes you filter the dataset manually, and sometimes automate it. Lasso can help determine which features contribute to predicting house prices and which can be safely ignored. That being said, Lasso is not the only way to do feature selection and not the best either.

Now, let's fit both Ridge and Lasso to our houses.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

DEG = 9
def poly_fit(estimator):
    m = make_pipeline(PolynomialFeatures(DEG, include_bias=False),
                      StandardScaler(), estimator).fit(size_tr.reshape(-1, 1), price_tr)
    return m, mse(m, size_tr, price_tr), mse(m, size_te, price_te)

m_none,  _, te_none  = poly_fit(LinearRegression())
m_ridge, _, te_ridge = poly_fit(Ridge(alpha=3.0))
m_lasso, _, te_lasso = poly_fit(Lasso(alpha=0.2, max_iter=300000))

w_none  = m_none.named_steps["linearregression"].coef_
w_ridge = m_ridge.named_steps["ridge"].coef_
w_lasso = m_lasso.named_steps["lasso"].coef_
zeros = lambda w: int((np.abs(w) < 1e-8).sum())

print(f"test MSEs:      no reg. = {te_none:.0f}      Ridge (L2) = {te_ridge:.0f}      Lasso (L1) = {te_lasso:.0f}")
print(f"\nweights set to exactly zero (out of {DEG})      "
      f"no reg. = {zeros(w_none)}      Ridge (L2) = {zeros(w_ridge)}      Lasso (L1) = {zeros(w_lasso)}")

grid = np.linspace(600, 3000, 300)
fits = [("no reg.", m_none, "tab:red"), ("Ridge (L2)", m_ridge, "tab:brown"), ("Lasso (L1)", m_lasso, "tab:purple")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.0), sharey=True)
for ax, (name, m, color) in zip(axes, fits):
    ax.scatter(size_te, price_te, color="tab:orange", alpha=0.2,  s=8,  zorder=2, label="test data")
    ax.scatter(size_tr, price_tr, color="tab:blue",  alpha=0.85, s=25, zorder=3, label="train data")
    ax.plot(grid, true_price(grid),               color="tab:green", lw=2, ls="--", zorder=4, label="true relationship")
    ax.plot(grid, m.predict(grid.reshape(-1, 1)), color=color,       lw=2,          zorder=5, label="model fit")
    ax.set_title(f"{name}\ntest MSE {mse(m, size_te, price_te):,.0f}")
    ax.set_xlabel("size")
axes[0].set_ylabel("price ($1000s)")
axes[0].set_ylim(0, 300)
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

labels = [f"$x^{k}$" for k in range(1, DEG + 1)]
idx = np.arange(DEG)
fig, ax = plt.subplots(figsize=(10.5, 4.2))
ax.bar(idx - 0.25, w_none,  width=0.25, label="no reg.",     color="tab:red")
ax.bar(idx,        w_ridge, width=0.25, label="Ridge (L2)",  color="tab:brown")
ax.bar(idx + 0.25, w_lasso, width=0.25, label="Lasso (L1)",  color="tab:purple")
clip = 1.3 * max(np.abs(w_ridge).max(), np.abs(w_lasso).max())
for i, v in enumerate(w_none):
    if abs(v) > clip:
        ax.annotate(f"{v:.0e}", xy=(i - 0.25, np.sign(v) * clip * 0.92), color="white",
                    ha="center", va=("top" if v > 0 else "bottom"), fontsize=8,
                    rotation=90, fontweight="bold")
ax.set_ylim(-clip, clip)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(idx); ax.set_xticklabels(labels)
ax.set_xlabel("polynomial term"); ax.set_ylabel("weight value")
ax.set_title("Lasso (L1) drives the high-degree weights to exactly zero, Ridge (L2) only shrinks them")
ax.legend()
plt.tight_layout()
plt.show()

Look at what regularization did. The unpenalized degree-9 fit overfit badly, with a test MSE in the thousands and high-degree weights growing crazy. Both penalties fixed it without changing the model's degree at all: Ridge brought the test MSE down to about 390 and Lasso to about 410.

The bar chart shows *how* they did it differently. Lasso set most of the nine weights to exactly zero (six of them here), keeping only the lowest-degree terms and switching the rest off. In effect it discovered on its own that a low-degree polynomial was enough. Ridge behaved differently. It shrank every weight smoothly toward zero but set none of them exactly to zero, so it still leans on all nine terms, just with small weights.

> **Side note:** Think of every data point as being made of two pieces. The first piece is **deterministic** (not random). It comes from a fixed rule that says how $y$ depends on $x$, and that rule is the green dashed curve. The second piece is random noise that gets added on top. The noise is different every single time and there is no pattern to it. A model learns by spotting patterns, so the best it can ever do is recover that fixed rule. The random part has no pattern to spot, which means it is unpredictable by definition, so no model can ever guess it no matter how clever it is. That is why even the perfect model one can ever train, the green curve itself, still has a nonzero MSE. It gets the deterministic part exactly right, and the error that is left over is just the random noise that it was never going to predict.

### Why L1 zeroes weights while L2 only shrinks them

The difference lies in how hard each penalty pulls a weight toward zero as that weight gets small. Remember that gradient descent nudges each weight against the derivative of the loss, so what matters here is the derivative of the penalty.

For L2 the penalty is $w^2$, whose derivative is $2w$. As a weight shrinks toward zero this pull fades away, since at $w = 0$ the derivative is also $0$. L2 eases off exactly when a weight is already small, so it shrinks weights but never quite finishes the job.

For L1 the penalty is $|w|$, whose derivative is $+1$ for any positive weight and $-1$ for any negative one. This pull is constant, just as strong no matter how tiny the weight is. So L1 keeps pushing a small, useless weight until it lands on exactly zero, and then holds it there. This is why Lasso switched off most of the features while Ridge merely made their weights smaller.

The plot below shows the two penalties and their pull, which is the derivative.

In [ ]:
w = np.linspace(-2, 2, 400)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(w, np.abs(w), color="tab:purple", lw=2.5, label="L1 penalty  |w|")
ax1.plot(w, w**2,      color="tab:brown",   lw=2.5, label="L2 penalty  w^2")
ax1.set_title("The penalty as a function of one weight")
ax1.set_xlabel("weight w"); ax1.set_ylabel("penalty")
ax1.axvline(0, color="black", lw=0.5); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(w, np.sign(w), color="tab:purple", lw=2.5, label="L1 pull = +/-1 (constant)")
ax2.plot(w, 2 * w,      color="tab:brown",   lw=2.5, label="L2 pull = 2w (fades near 0)")
ax2.axvline(0, color="black", lw=0.5); ax2.axhline(0, color="black", lw=0.5)
ax2.set_title("How hard each penalty pulls a weight toward zero")
ax2.set_xlabel("weight w"); ax2.set_ylabel("pull (slope of penalty)")
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Lambda is a hyperparameter you have to tune

Regularization hands us the dial $\lambda$, and like the learning rate from Notebook 02 it is a hyperparameter, meaning we set it ourselves rather than letting gradient descent learn it. Its value fixes the trade-off, and both extremes are bad.

When $\lambda$ is too small, near zero, there is almost no penalty, so the model keeps its large high-degree weights and overfits. When $\lambda$ is too large, the penalty dominates and crushes every weight toward zero, including the low-degree terms that we genuinely need, so the model underfits. Below, we train a degree-9 polynomial with varying $\lambda$ to see this in action. It can be clearly seen that when $\lambda$ is small, the test MSE is high due to overfitting, and as $\lambda$ increases, the test MSE decreases until it reaches a minimum point. Beyond that point, as $\lambda$ continues to increase, the test MSE starts to rise again because regularization is now too strong and starts hurting the model.

In [ ]:
lambdas = np.logspace(-3, 1.5, 30)
train_path, test_path = [], []
for a in lambdas:
    m, tr, te = poly_fit(Ridge(alpha=a, max_iter=300000))
    train_path.append(tr)
    test_path.append(te)
best = int(np.argmin(test_path))

plt.figure(figsize=(8, 4.6))
plt.plot(lambdas, train_path, "o-", color="tab:blue",   ms=3, label="train MSE")
plt.plot(lambdas, test_path,  "o-", color="tab:orange", ms=3, label="test MSE")
plt.axvline(lambdas[best], color="gray", ls="--", label=f"best lambda ~ {lambdas[best]:.2g}")
plt.xscale("log")
plt.xlabel("regularization strength  lambda")
plt.ylabel("MSE")
plt.legend()
plt.title("Test error is lowest at a moderate lambda; too small overfits, too large underfits")
plt.tight_layout()
plt.show()

## Summary

- Feature engineering lets us hand a model many features, such as hand-crafted polynomial features built from a single feature, and the model learns one weight per feature. The danger is that not every feature matters. With limited data a model may overfit by putting large weight on features it does not need, as the high-degree polynomial terms did.
- Regularization fights this by adding a penalty on the size of the weights to the loss, $\mathcal{L} = \text{MSE} + \lambda \cdot \text{penalty}(w)$, so a feature only keeps its weight if it earns it.
- L2, or Ridge, penalizes $\sum_j w_j^2$ and shrinks all weights smoothly toward zero, but not exactly zero.
- L1, or Lasso, penalizes $\sum_j |w_j|$, and its constant pull drives unneeded weights to exactly zero. That deletes the useless features outright, which guards against overfitting and doubles as feature selection.
- The weight decay coefficient $\lambda$ is another hyperparameter we need to tune. Too small overfits, too large underfits, so we tune it on the validation set.